<a href="https://colab.research.google.com/github/AvirupVIP/SIGNIFY---Signature-Forgery-Detection-System/blob/master/MobileNet_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle
!pip install opencv-python
!pip install scikit-learn

In [2]:
import os
import json

kaggle_config = {
  "username":"sandandipto",
  "key":"KGAT_d5a3d64f806d070d376a3862e1c6f371"
}

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_config, f)

!chmod 600 /root/.kaggle/kaggle.json

In [3]:
!kaggle datasets download -d shreelakshmigp/cedardataset

Dataset URL: https://www.kaggle.com/datasets/shreelakshmigp/cedardataset
License(s): unknown
 55% 133M/242M [00:00<00:00, 1.39GB/s]
100% 242M/242M [00:00<00:00, 615MB/s] 


In [4]:
!unzip -q cedardataset.zip


In [5]:
!pip install opencv-python scikit-learn tqdm

In [6]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split

In [7]:
genuine_path = "signatures/full_org"
forged_path = "signatures/full_forg"

In [8]:
IMG_SIZE = 224

def preprocess_signature(path):

    img = cv2.imread(path)

    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(
        gray,0,255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    kernel = np.ones((3,3),np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    coords = cv2.findNonZero(opening)

    if coords is None:
        return None

    x,y,w,h = cv2.boundingRect(coords)
    crop = opening[y:y+h, x:x+w]

    resized = cv2.resize(crop,(IMG_SIZE,IMG_SIZE))

    normalized = resized/255.0

    normalized = np.stack((normalized,)*3, axis=-1)

    return normalized

In [9]:
images = []
labels = []

print("Loading Genuine Signatures...")

for img in tqdm(os.listdir(genuine_path)):

    if img.endswith(".png"):

        path = os.path.join(genuine_path,img)

        data = preprocess_signature(path)

        if data is not None:
            images.append(data)
            labels.append(1)


print("Loading Forged Signatures...")

for img in tqdm(os.listdir(forged_path)):

    if img.endswith(".png"):

        path = os.path.join(forged_path,img)

        data = preprocess_signature(path)

        if data is not None:
            images.append(data)
            labels.append(0)


X = np.array(images)
y = np.array(labels)

print("Dataset Shape:",X.shape)

Loading Genuine Signatures...


100%|██████████| 1321/1321 [00:09<00:00, 139.51it/s]


Loading Forged Signatures...


100%|██████████| 1321/1321 [00:10<00:00, 127.34it/s]


Dataset Shape: (2640, 224, 224, 3)


In [10]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [11]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(256,activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(1,activation='sigmoid')(x)

model = Model(inputs=base_model.input,outputs=output)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [12]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [13]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'],label='Train Accuracy')
plt.plot(history.history['val_accuracy'],label='Validation Accuracy')
plt.title("Accuracy Graph")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'],label='Train Loss')
plt.plot(history.history['val_loss'],label='Validation Loss')
plt.title("Loss Graph")
plt.legend()

plt.show()

In [ ]:
loss,accuracy = model.evaluate(X_test,y_test)

print("Test Accuracy:",accuracy)

In [ ]:
def predict_signature(path):

    img = preprocess_signature(path)

    img = np.expand_dims(img,axis=0)

    prob = model.predict(img)[0][0]

    similarity_score = prob

    if prob > 0.5:
        print("Prediction: Genuine")
    else:
        print("Prediction: Forged")

    print("Similarity Score:",similarity_score)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
predict_signature("test_signature.png")